# SMT — Horizontal Alignment: Plan View

This notebook gives a visual introduction to SMT's **horizontal alignment engine**.
It loads the 30-element validation alignment from `tests/golden/tables.json`,
traces every element using `calculate_point_on_element`, and renders a plan-view
plot coloured by element type (tangent / circular arc / transition spiral).

> **Prerequisite:** `pip install -e '.[notebook]'` from the repo root, then restart the kernel.
> Or `pip install matplotlib` directly.

## 1  The element model

SMT represents a horizontal alignment as an **ordered list of `Element` records**.
Every element carries its own geometry in a single unified struct:

| field | meaning |
|---|---|
| `sta_start`, `sta_end` | chainage range along the centre line (metres) |
| `n`, `e` | entry grid coordinates (Northing, Easting) |
| `az` | entry tangent azimuth — radians, WCB convention (north = 0, clockwise) |
| `k_in`, `k_out` | signed curvature k = 1/R at entry and exit (+  = right turn, − = left) |
| `trans` | transition shape: CLOTHOID / BLOSS / COSINE / SINE |

Three element *types* arise naturally from the curvature pair:

| type | `k_in` | `k_out` | note |
|---|---|---|---|
| **Tangent** (T) | 0 | 0 | straight line — curvature zero on both ends |
| **Circular arc** (C) | k | k | constant radius, k₁ = k₂ |
| **Spiral** (SPIN / SPOUT) | k₁ | k₂ | k₁ ≠ k₂, curvature changes smoothly |

### Key invariant: exit state = next entry state

The **exit state** of element *n* (position + tangent direction) equals the
**entry state** of element *n+1* exactly.  This makes the full alignment
seamless and allows each element to be checked independently.

> *"Make illegal states unrepresentable."*  
> A straight line is just k = 0, not a special case — this single unified
> model eliminates most type-dispatch logic in the engine.
> (see `docs/lesson.md` §2.1 for the full reasoning)

### Two core operations

| function | direction | use |
|---|---|---|
| `calculate_station_to_coord(elements, sta, offset)` | station → N, E | set-out, surface modelling |
| `calculate_coord_to_station(elements, n, e)` | N, E → station, offset | as-built survey, cross-sections |

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Make the SMT source tree importable when running from notebooks/
# (unnecessary if the package has been installed with pip install -e .)
_src = Path("../src").resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from smt.alignment import parse_alignment_table, calculate_point_on_element

In [ ]:
# Locate the golden table whether Jupyter was started from
# the repo root or the notebooks/ subdirectory.
for _candidate in [
    Path("../tests/golden/tables.json"),
    Path("tests/golden/tables.json"),
]:
    if _candidate.exists():
        GOLDEN = _candidate
        break
else:
    raise FileNotFoundError("tables.json not found — start Jupyter from the repo root")

data = json.loads(GOLDEN.read_text(encoding="utf-8"))
elements = parse_alignment_table(data["elements"])
controls = data["controls"]

print(f"Loaded {len(elements)} elements and {len(controls)} control points")
print(f"Station range: {elements[0].sta_start:.3f} — {elements[-1].sta_end:.3f} m")

type_counts: dict[str, int] = {}
for el in elements:
    type_counts[el.type] = type_counts.get(el.type, 0) + 1
print("Element type counts:", type_counts)

## 2  Plan view

Each element is sampled at 60 evenly-spaced arc distances using
`calculate_point_on_element`.  For spirals this integrates dx/ds = cos θ(s)
and dy/ds = sin θ(s) numerically (Simpson's rule, 48 sub-intervals).

Axes follow the **survey convention**: Northing on the vertical axis
(north-up), Easting on the horizontal axis.  Colours indicate element type.
Control points (PC, PT, TS, SC, CS, ST, PCC, BP, EP) are marked with
their standard labels.

In [ ]:
COLORS: dict[str, str] = {
    "T":     "#1976D2",   # blue   — tangent
    "C":     "#E64A19",   # orange — circular arc
    "SPIN":  "#388E3C",   # green  — spiral in  (curvature ramps up)
    "SPOUT": "#7B1FA2",   # purple — spiral out (curvature ramps down)
}
TYPE_LABELS: dict[str, str] = {
    "T":     "Tangent (T)",
    "C":     "Circular arc (C)",
    "SPIN":  "Spiral in (SPIN)",
    "SPOUT": "Spiral out (SPOUT)",
}
N_STEPS = 60  # sample points per element

fig, ax = plt.subplots(figsize=(9, 13))
ax.set_aspect("equal")
ax.set_xlabel("Easting  E (m)", labelpad=8)
ax.set_ylabel("Northing  N (m)", labelpad=8)
ax.set_title(
    "Horizontal Alignment — Plan View\n"
    f"{len(elements)} elements  |  "
    f"STA {elements[0].sta_start:.0f} – {elements[-1].sta_end:.0f} m",
    pad=14,
)

# --- element traces, coloured by type ---
plotted_types: set[str] = set()
for el in elements:
    L = el.sta_end - el.sta_start
    pts_e: list[float] = []
    pts_n: list[float] = []
    for i in range(N_STEPS + 1):
        state = calculate_point_on_element(el, L * i / N_STEPS)
        pts_n.append(state.n)
        pts_e.append(state.e)
    color = COLORS.get(el.type, "#607D8B")
    label = TYPE_LABELS.get(el.type) if el.type not in plotted_types else None
    plotted_types.add(el.type)
    ax.plot(pts_e, pts_n, color=color, linewidth=2.5,
            label=label, solid_capstyle="round")

# --- control-point markers ---
ax.scatter(
    [c["e"] for c in controls],
    [c["n"] for c in controls],
    color="black", s=28, zorder=5, label="Control points",
)

# Labels with alternating left/right offset to reduce overlap
for i, c in enumerate(controls):
    dx = 14 if i % 2 == 0 else -14
    ax.annotate(
        c["name"],
        xy=(c["e"], c["n"]),
        xytext=(c["e"] + dx, c["n"] + 8),
        fontsize=7.5,
        ha="left" if dx > 0 else "right",
        va="bottom",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75),
    )

ax.legend(loc="upper right", framealpha=0.9, fontsize=9)
ax.grid(True, alpha=0.2, linestyle="--")
plt.tight_layout()
plt.show()